In [3]:
"""
CNN Embedding Extraction — Coral Bleaching Project
====================================================
Goal: extract a 256-dim CNN embedding for EVERY DAY in your dataset
(not per 7-day sequence). This is the right granularity for fusion later,
because your friend's NLP embeddings are SEASONAL — you'll broadcast each
season's NLP vector across its days, and this gives you one CNN vector per
day to attach it to.

Paste each "CELL" below into a separate Colab cell, in order.
"""

# ══════════════════════════════════════════════════════════════════════════
# CELL 1 — Get your data into Colab
# ══════════════════════════════════════════════════════════════════════════
# OPTION A (recommended): Mount Google Drive — your CNN folder is already there
from google.colab import files
import zipfile, os

print("Click 'Choose Files' and select 3_dataset.zip ...")
uploaded = files.upload()

with zipfile.ZipFile('3_dataset.zip', 'r') as z:
    z.extractall('.')
print('Unzipped!')

DATASET_DIR = '/content/3_dataset'
SAVE_DIR    = '/content/4_models'
RESULTS_DIR = '/content/5_results'

os.makedirs(SAVE_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Dataset ready!')
print('Train files:', len(os.listdir(f'{DATASET_DIR}/train')))
print('Test files: ', len(os.listdir(f'{DATASET_DIR}/test')))

Click 'Choose Files' and select 3_dataset.zip ...


Saving 3_dataset.zip to 3_dataset.zip
Unzipped!
Dataset ready!
Train files: 1915
Test files:  727


In [8]:
print("Click 'Choose Files' and select cnn_lstm_dropout_best.pth ...")
uploaded = files.upload()

Click 'Choose Files' and select cnn_lstm_dropout_best.pth ...


Saving cnn_lstm_dropout_best.pth to cnn_lstm_dropout_best.pth


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 2 — Set paths
# ══════════════════════════════════════════════════════════════════════════
import os

DATASET_DIR = '/content/3_dataset'
MODEL_PATH  = '/content/cnn_lstm_dropout_best.pth'
RESULTS_DIR = '/content/5_results'

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Dataset dir exists:", os.path.exists(DATASET_DIR))
print("Model file exists:", os.path.exists(MODEL_PATH))
print("Train files:", len(os.listdir(f'{DATASET_DIR}/train')))
print("Test files: ", len(os.listdir(f'{DATASET_DIR}/test')))

Dataset dir exists: True
Model file exists: True
Train files: 1915
Test files:  727


In [10]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 3 — Imports
# ══════════════════════════════════════════════════════════════════════════
import re
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)



Using device: cuda


In [11]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4 — Model definition (must match training exactly)
# ══════════════════════════════════════════════════════════════════════════
class CNNFeatureExtractor(nn.Module):
    def __init__(self, in_channels=4, feature_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32,  kernel_size=3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64,           kernel_size=3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(64, 128,          kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(128, 256,         kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Linear(256 * 4 * 4, feature_dim)

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class CoralCNNLSTM(nn.Module):
    """Full model — needed so state_dict keys line up. We'll only use .cnn"""
    def __init__(self, feature_dim=256, hidden_dim=128, num_layers=2, num_classes=5):
        super().__init__()
        self.cnn  = CNNFeatureExtractor(in_channels=4, feature_dim=feature_dim)
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=hidden_dim,
                            num_layers=num_layers, batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64, num_classes)
        )

    def forward(self, x):
        batch, seq_len, C, H, W = x.shape
        cnn_out = [self.cnn(x[:, t]) for t in range(seq_len)]
        cnn_out     = torch.stack(cnn_out, dim=1)
        lstm_out, _ = self.lstm(cnn_out)
        return self.classifier(lstm_out[:, -1, :])


# Load full model (so the .cnn submodule gets the trained weights correctly)
full_model = CoralCNNLSTM().to(device)
full_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
full_model.eval()

cnn_extractor = full_model.cnn   # ← this is what we'll run on each day's image
print('CNN feature extractor loaded. Output dim: 256')


CNN feature extractor loaded. Output dim: 256


In [12]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5 — Build a daily file index (date, split, label, path)
# ══════════════════════════════════════════════════════════════════════════
records = []

for split in ['train', 'test']:
    split_dir  = f'{DATASET_DIR}/{split}'
    labels_csv = f'{DATASET_DIR}/{split}_labels.csv'

    labels_df = pd.read_csv(labels_csv)
    labels_df['date'] = pd.to_datetime(labels_df['date'])
    label_map = dict(zip(labels_df['date'].dt.strftime('%Y-%m-%d'), labels_df['label']))

    for f in sorted(glob.glob(f'{split_dir}/stacked_*.npy')):
        m = re.match(r'stacked_(\d{4})_(\d{2})_(\d{2})\.npy', os.path.basename(f))
        if not m:
            continue
        y, mo, d = m.groups()
        date_str = f'{y}-{mo}-{d}'
        records.append({
            'date':  date_str,
            'split': split,
            'label': label_map.get(date_str, np.nan),
            'path':  f
        })

file_index = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
print(f'Total days found: {len(file_index)}')
print(file_index.head())
print(file_index['split'].value_counts())

Total days found: 2642
         date  split  label                                             path
0  2018-01-01  train      2  /content/3_dataset/train/stacked_2018_01_01.npy
1  2018-01-02  train      2  /content/3_dataset/train/stacked_2018_01_02.npy
2  2018-01-03  train      2  /content/3_dataset/train/stacked_2018_01_03.npy
3  2018-01-04  train      2  /content/3_dataset/train/stacked_2018_01_04.npy
4  2018-01-05  train      2  /content/3_dataset/train/stacked_2018_01_05.npy
split
train    1915
test      727
Name: count, dtype: int64


In [13]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 — Extract CNN embeddings (one 256-dim vector per day)
# ══════════════════════════════════════════════════════════════════════════
embeddings = np.zeros((len(file_index), 256), dtype=np.float32)

with torch.no_grad():
    for i, row in file_index.iterrows():
        arr = np.load(row['path'])                              # (4, H, W)
        x   = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 4, H, W)
        emb = cnn_extractor(x)                                   # (1, 256)
        embeddings[i] = emb.cpu().numpy().squeeze(0)

        if i % 200 == 0:
            print(f'{i}/{len(file_index)} done')

print('Embeddings shape:', embeddings.shape)

0/2642 done
200/2642 done
400/2642 done
600/2642 done
800/2642 done
1000/2642 done
1200/2642 done
1400/2642 done
1600/2642 done
1800/2642 done
2000/2642 done
2200/2642 done
2400/2642 done
2600/2642 done
Embeddings shape: (2642, 256)


In [14]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 — Save everything
# ══════════════════════════════════════════════════════════════════════════
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. Raw embeddings as .npy (for fast loading later)
np.save(f'{RESULTS_DIR}/cnn_embeddings_daily.npy', embeddings)

# 2. Combined CSV: date, split, label + 256 embedding columns
#    This is the file you'll merge with the seasonal NLP embeddings.
emb_cols = [f'emb_{i}' for i in range(256)]
emb_df   = pd.DataFrame(embeddings, columns=emb_cols)

combined = pd.concat([file_index[['date', 'split', 'label']].reset_index(drop=True), emb_df], axis=1)
combined.to_csv(f'{RESULTS_DIR}/cnn_embeddings_daily.csv', index=False)

print(f'Saved to {RESULTS_DIR}/')
print(combined.iloc[:, :6].head())   # preview first few columns

Saved to /content/5_results/
         date  split  label     emb_0     emb_1     emb_2
0  2018-01-01  train      2  0.499173 -0.362648  0.173684
1  2018-01-02  train      2  0.565268 -0.395259  0.076485
2  2018-01-03  train      2  0.313584 -0.530867  0.097723
3  2018-01-04  train      2  0.399997 -0.144843  0.236500
4  2018-01-05  train      2  0.341071 -0.072196  0.215122


In [15]:
# ── Optional: download to your laptop ──
from google.colab import files
files.download(f'{RESULTS_DIR}/cnn_embeddings_daily.csv')
files.download(f'{RESULTS_DIR}/cnn_embeddings_daily.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>